In [ ]:
# https://www.kaggle.com/datasets/wyattowalsh/basketball/data
# https://github.com/mpope9/nba-sql/blob/master/image/NBA-ER.jpg

In [1]:
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
import warnings
import re
import psycopg2

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
options = webdriver.FirefoxOptions()
options.add_argument('-headless')
driver = webdriver.Firefox(options = options)

The geckodriver version (0.33.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (134.0.0.3375); currently, geckodriver 0.35.0 is recommended for firefox 134.*, so it is advised to delete the driver in PATH and retry


In [58]:
def getTableIDS(url):
    driver.get(url)
    tables_id = driver.find_elements(By.XPATH, "//table[@id]")
    list_id_tables = []
    for table in tables_id:
        table_id = table.get_attribute("id")
        list_id_tables.append(table_id)
    return list_id_tables

In [ ]:
def delete_unnamed_columns(df):
    df = df.loc[:, ~df.columns.str.contains('Unnamed')]
    return df

In [ ]:
def get_keys_dictionary(diccionario):
    keys = set(diccionario.keys())
    for values in diccionario.values():
        if isinstance(values, dict):
            keys.update(get_keys_dictionary(values))
    return keys

In [ ]:
def check_missing_values(dictionary):
    for key, value in dictionary.items():
        if isinstance(value, dict):
            print(f"Recorriendo diccionario bajo la clave '{key}':")
            check_missing_values(value)  
        elif isinstance(value, pd.DataFrame): 
            print(f"Revisando DataFrame bajo la clave '{key}':")
            
            if value.isnull().values.any():
                print("¡Hay valores nulos en el DataFrame!")
                print(value)
            
            unnamed_columns = [col for col in value.columns if 'Unnamed' in col]
            if unnamed_columns:
                print(f"¡El DataFrame tiene columnas 'Unnamed': {unnamed_columns}")

            empty_columns = [col for col in value.columns if value[col].empty]
            if empty_columns:
                print(f"¡El DataFrame tiene columnas vacías: {empty_columns}")

In [ ]:
# NBA Standings que es como quedó la season con todos los equipos
years = list(range(2022, 2023))
dictionary_of_teams = {}
# Itera a través de cada identificador de tabla y guarda en un DataFrame
dataframes = []
for year in years:
    dictionary_of_teams[year] = {}
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_standings.html'
    table_ids = getTableIDS(url)
    time.sleep(3)
    for table_id in table_ids:
        dictionary_of_teams[year][table_id] = {}
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        if table_id == 'expanded_standings':
            new_header = df.iloc[0]
            df = df[1:]
            df.columns = new_header
            df.reset_index(drop=True, inplace=True)
            dictionary_of_teams[year][table_id] = df
        else:    
            dictionary_of_teams[year][table_id] = df

In [ ]:
# De aqui para arriba tenemos las estadisticas generales de los equipos
# De aqui para abajo sacaremos las estadisticas de los jugadores por equipo

In [4]:
years = list(range(2022, 2023))
dictionary_of_players = {}
dictionary_of_players_playoffs = {}
teams_NBA_list = ['ATL']
# teams_NBA_list =  ['ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND','LAC','LAL','MEM','MIA','MIL','MIN','NOP', 'NYK','OKC', 'ORL','PHI', 
#                    'PHO', 'POR','SAC','SAS','TOR','UTA','WAS']

for team in teams_NBA_list:
    dictionary_of_players[team] = {}
    dictionary_of_players_playoffs[team] = {}
    for year in years:
        dictionary_of_players[team][year] = {}
        dictionary_of_players_playoffs[team][year] = {}
        url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'
        table_ids = getTableIDS(url)
        time.sleep(3)
        for table_id in table_ids:
            if 'playoffs' in table_id:
                dictionary_of_players_playoffs[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if  table_id == 'playoffs_pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players_playoffs[team][year][table_id] = df
            else:
                dictionary_of_players[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if table_id == 'adj_shooting' or table_id == 'shooting' or table_id == 'pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players[team][year][table_id] = df
        

In [3]:
class PlayerScraper:
        def __init__(self):
            self.url = 'https://www.basketball-reference.com/teams/',
            self.teams_NBA_list = 'ATL',
            self.years = '2024',
    
        def get_players_team_year(self):
            dictionary_of_teams = {}
            for nba_team in self.teams_NBA_list:
                dictionary_of_teams[nba_team] = {} 
                for year in self.years: 
                    dictionary_of_teams[nba_team][year] = [] 
                    url = f'https://www.basketball-reference.com/teams/{nba_team}/{year}.html' 
                    response = requests.get(url) 
                    soup = BeautifulSoup(response.content, 'html.parser') 
                    table = soup.find('table', {'id': 'advanced'}) 
                    if table: 
                        headers = [th.text.strip() for th in table.find('thead').find_all('th')] 
                        rows = [ 
                            {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(['th', 'td']))} 
                            for tr in table.find('tbody').find_all('tr') 
                        ] 
                        dictionary_of_teams[nba_team][year] = rows 
            return dictionary_of_teams

In [3]:
class TeamScraper:
    def __init__(self):
        self.url = 'https://www.basketball-reference.com/teams/'

    def get_team_table(self):
        response = requests.get(self.url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            table = soup.find('table', {'id': 'teams_active'})
            if table:
                headers = [th.text.strip() for th in table.find('thead').find_all('th')]
                headers = ['Team' if header == 'Franchise' else header for header in headers]
                
                rows = [
                    {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(['th', 'td']))}
                    for tr in table.find('tbody').find_all('tr', class_='full_table')
                ]
                
                return rows
            else:
                raise ValueError("Could not find the table with id 'teams_active'")
        else:
            raise Exception(f"Failed to retrieve data, status code: {response.status_code}")

In [4]:
team = TeamScraper()
team_st = team.get_team_table()

In [4]:
scraper_player = PlayerScraper()
players_data_by_team = scraper_player.get_players_team_year()

In [5]:
for team, year in dictionary_of_players.items():
    for year, tables in year.items():
        print(tables.keys())


# DE AQUI VER QUE TABLA INTERESA, E INTENTAR VER COMO ESCTRUCTURAR LA BASE DE DATOS
# PERO ANTES DE NADA CREAR TABLAS CON LOS DATOS QUE QUERAMOS

dict_keys(['roster', 'team_and_opponent', 'team_misc', 'per_game', 'totals', 'per_minute', 'per_poss', 'advanced', 'adj_shooting', 'shooting', 'pbp', 'salaries2'])


In [6]:
class PlayerScraper:
        def __init__(self):
            self.url = 'https://www.basketball-reference.com/teams/',
            self.teams_NBA_list = 'ATL',
            self.years = '2024',
        
        def get_team_regular_season_results(self):
            dictionary_of_teams = {}
            for nba_team in self.teams_NBA_list:
                dictionary_of_teams[nba_team] = {}
                for year in self.years:
                    dictionary_of_teams[nba_team][year] = []
                    url = f'https://www.basketball-reference.com/teams/{nba_team}/{year}.html'
                    response = requests.get(url)
                    soup = BeautifulSoup(response.content, 'html.parser')
                    table = soup.find('table', {'id': 'advanced'})
                    if table:
                        headers = [th['data-stat'] for th in table.find('thead').find_all('th')]
                        for tr in table.find('tbody').find_all('tr'):
                            # Ignorar los tr que tienen la clase 'thead'
                            if 'thead' in tr.get('class', []):
                                continue
                            # Ignorar filas que tienen un th con el valor del primer header
                            if tr.find('th') and tr.find('th').text.strip() == headers[0]:
                                continue
                            row_data = {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(['th', 'td']))}
                            dictionary_of_teams[nba_team][year].append(row_data)
            return dictionary_of_teams


In [9]:
##test para probar si funciona total_stats
class PlayerScraper:
        def __init__(self):
            self.url = 'https://www.basketball-reference.com/teams/',
            self.teams_NBA_list = 'ATL',
            self.years = '2024',
        
        def get_team_regular_season_results(self):
            dictionary_of_teams = {}
            for nba_team in self.teams_NBA_list:
                dictionary_of_teams[nba_team] = {}
                for year in self.years:
                    dictionary_of_teams[nba_team][year] = []
                    url = f'https://www.basketball-reference.com/teams/{nba_team}/{year}.html'
                    response = requests.get(url)
                    # if not response.content:
                    #     print(f"No content found for {nba_team} in {year}.")
                    #     return []
                    #  # Save and inspect the response
                    # with open("debug_totals_stats.html", "w", encoding="utf-8") as file:
                    #     file.write(response.content.decode("utf-8"))
                        
                    soup = BeautifulSoup(response.content, 'html.parser')
                    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
                    for comment in comments:
                        if "advanced" in comment:  # Check if the desired table is inside the comment
                            comment_soup = BeautifulSoup(comment, "html.parser")  # Parse the commented content
                            table = comment_soup.find("table", {"id": "advanced"})
                            if table:
                                print(f"Found advanced table for {nba_team} in {year}")
                                headers = [th.text.strip() for th in table.find("thead").find_all("th")]
                                rows = [
                                    {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(["th", "td"]))}
                                    for tr in table.find("tbody").find_all("tr")
                                ]
                                return rows

                    print(f"Advanced table not found for {nba_team} in {year}.")
                    return []

In [10]:
scraper_player = PlayerScraper()
team_advanced = scraper_player.get_team_regular_season_results()

Found advanced table for ATL in 2024


In [11]:
team_advanced

[{'Rk': '1',
  'Player': 'Dejounte Murray',
  'Age': '27',
  'Pos': 'SG',
  'G': '78',
  'GS': '78',
  'MP': '2783',
  'PER': '17.7',
  'TS%': '.555',
  '3PAr': '.379',
  'FTr': '.179',
  'ORB%': '2.3',
  'DRB%': '14.4',
  'TRB%': '8.1',
  'AST%': '27.9',
  'STL%': '1.9',
  'BLK%': '0.8',
  'TOV%': '11.3',
  'USG%': '26.6',
  'OWS': '3.3',
  'DWS': '1.6',
  'WS': '4.9',
  'WS/48': '.084',
  'OBPM': '2.3',
  'DBPM': '-0.6',
  'BPM': '1.7',
  'VORP': '2.6',
  'Awards': ''},
 {'Rk': '2',
  'Player': 'Bogdan Bogdanović',
  'Age': '31',
  'Pos': 'SG',
  'G': '79',
  'GS': '33',
  'MP': '2401',
  'PER': '14.7',
  'TS%': '.569',
  '3PAr': '.583',
  'FTr': '.149',
  'ORB%': '2.3',
  'DRB%': '10.3',
  'TRB%': '6.2',
  'AST%': '14.9',
  'STL%': '1.9',
  'BLK%': '1.0',
  'TOV%': '8.7',
  'USG%': '22.3',
  'OWS': '2.8',
  'DWS': '1.2',
  'WS': '3.9',
  'WS/48': '.079',
  'OBPM': '0.9',
  'DBPM': '-0.7',
  'BPM': '0.2',
  'VORP': '1.4',
  'Awards': '6MOY-5'},
 {'Rk': '3',
  'Player': 'Saddiq Bey',


In [9]:
for team, year in team_advanced.items():
    print(team)

ATL


In [68]:
team_advanced['ATL']['2024'][20]

In [8]:
for nba_team, games in player_advanced.items():
    for year, games in games.items():
        for stats in games:
            print(stats.keys())

In [1]:
team_abbrev = {
    'Atlanta Hawks': 'ATL',
    'Boston Celtics': 'BOS',
    'Brooklyn Nets': 'BRK',
    'Charlotte Hornets': 'CHO',
    'Chicago Bulls': 'CHI',
    'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL',
    'Denver Nuggets': 'DEN',
    'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW',
    'Houston Rockets': 'HOU',
    'Indiana Pacers': 'IND',
    'Los Angeles Clippers': 'LAC',
    'Los Angeles Lakers': 'LAL',
    'Memphis Grizzlies': 'MEM',
    'Miami Heat': 'MIA',
    'Milwaukee Bucks': 'MIL',
    'Minnesota Timberwolves': 'MIN',
    'New Orleans Pelicans': 'NOP',
    'New York Knicks': 'NYK',
    'Oklahoma City Thunder': 'OKC',
    'Orlando Magic': 'ORL',
    'Philadelphia 76ers': 'PHI',
    'Phoenix Suns': 'PHO',
    'Portland Trail Blazers': 'POR',
    'Sacramento Kings': 'SAC',
    'San Antonio Spurs': 'SAS',
    'Toronto Raptors': 'TOR',
    'Utah Jazz': 'UTA',
    'Washington Wizards': 'WAS'
}

In [ ]:
#celda de limpieza de datos
dictionary_of_players['ATL'][2022]['roster'] = dictionary_of_players['ATL'][2022]['roster'].drop(columns=['Unnamed: 6'])
dictionary_of_players['ATL'][2022]['team_and_opponent'] = dictionary_of_players['ATL'][2022]['team_and_opponent'].rename(columns={'Unnamed: 0': ''})
dictionary_of_players['ATL'][2022]['team_misc'].columns = dictionary_of_players['ATL'][2022]['team_misc'].iloc[0]
dictionary_of_players['ATL'][2022]['team_misc'] = dictionary_of_players['ATL'][2022]['team_misc'][1:]
dictionary_of_players['ATL'][2022]['per_poss'] = dictionary_of_players['ATL'][2022]['per_poss'] .drop(columns=['Unnamed: 27'])
dictionary_of_players['ATL'][2022]['advanced'] = dictionary_of_players['ATL'][2022]['advanced'].drop(columns=['Unnamed: 17', 'Unnamed: 22'])